In [7]:
from datasets import load_dataset, DatasetDict

hf_nature_ds = load_dataset("mertcobanov/nature-dataset")
split = hf_nature_ds["train"].train_test_split(test_size=0.2, seed=42)
hf_nature_ds = DatasetDict({
    "train": split["train"],
    "validation": split["test"]
})

hf_cityscapes_ds: DatasetDict = load_dataset("Oursel/cityscapes")
hf_nature_ds, hf_cityscapes_ds

cityscapes.zip:   0%|          | 0.00/12.0G [00:00<?, ?B/s]

mapiliary vista.zip:   0%|          | 0.00/25.6G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

(DatasetDict({
     train: Dataset({
         features: ['image', 'caption'],
         num_rows: 40000
     })
     validation: Dataset({
         features: ['image', 'caption'],
         num_rows: 10000
     })
 }),
 DatasetDict({
     train: Dataset({
         features: ['image'],
         num_rows: 114974
     })
 }))

In [6]:
NOT_TRAVERSIBLE_KEYWORDS = {
    # Water
    "water", "ocean", "river", "lake", "stream", "pond",
    "waterfall", "flood", "swamp", "marsh", "sea",
    # Hazardous terrain
    "cliff", "steep", "ravine", "gorge", "canyon",
    "chasm", "precipice",
    # Dense vegetation
    "thicket", "bramble",
    # Other
    "lava", "quicksand", "mud", "snow", "ice", "glacier",
}

def label_traversibility(example):
    caption = example["caption"].lower()
    # Tokenize on word boundaries to avoid partial matches
    words = set(caption.replace(",", " ").replace(".", " ").split())
    is_not_traversible = bool(words & NOT_TRAVERSIBLE_KEYWORDS)
    return {"label": 0 if is_not_traversible else 1}

hf_nature_ds = hf_nature_ds.map(label_traversibility)
hf_nature_ds = hf_nature_ds.remove_columns(["caption"])
hf_nature_ds

Map:   0%|          | 0/40000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 40000
    })
    validation: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

In [ ]:
import os
import tensorflow as tf
from tensorflow.data import Dataset as TFDataset
num_cpus = os.cpu_count() if os.cpu_count() is not None else 1
# Now, split into training and validation tensorflow datasets
train_cityscapes_ds: TFDataset = hf_cityscapes_ds["train"].to_tf_dataset(
    columns="image",
    label_cols="label",
    batch_size=32,
    shuffle=True,
    num_workers=num_cpus
)
validation_cityscapes_ds: TFDataset = hf_cityscapes_ds["validation"].to_tf_dataset(
    columns="image",
    label_cols="label",
    batch_size=32,
    shuffle=False,
    num_workers=num_cpus
)

train_nature_ds: TFDataset = hf_nature_ds["train"].to_tf_dataset(
    columns="image",
    label_cols="label",
    batch_size=32,
    shuffle=True,
    num_workers=num_cpus
)
validation_rugd_ds: TFDataset = hf_nature_ds["validation"].to_tf_dataset(
    columns="image",
    label_cols="label",
    batch_size=32,
    shuffle=False,
    num_workers=num_cpus
)
train_cityscapes_ds, validation_cityscapes_ds, train_natue_ds, validation_rugd_ds

In [ ]:
# Cast all dataset image dtypes to be tf.uint8, which is the standard...
train_cityscapes_ds: TFDataset = train_cityscapes_ds.map(lambda img, lbl: (tf.cast(img, tf.uint8), lbl))
validation_cityscapes_ds: TFDataset = validation_cityscapes_ds.map(lambda img, lbl: (tf.cast(img, tf.uint8), lbl))
train_rugd_ds: TFDataset = nature_rugd_ds.map(lambda img, lbl: (tf.cast(img, tf.uint8), lbl))
validation_rugd_ds: TFDataset = validation_rugd_ds.map(lambda img, lbl: (tf.cast(img, tf.uint8), lbl))
train_cityscapes_ds, validation_cityscapes_ds, train_rugd_ds, validation_rugd_ds

In [ ]:
import matplotlib.pyplot as plt
image, label = next(iter(train_cityscapes_ds.unbatch().take(1)))

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Original image
axes[0].imshow(image.numpy().astype("uint8"))
axes[0].set_title("Image")
axes[0].axis("off")

# Segmentation mask
axes[1].imshow(label.numpy(), cmap="tab20")  # tab20 gives distinct colors per class
axes[1].set_title("Segmentation Mask")
axes[1].axis("off")

plt.show()